# DCGAN for Minority-Class Augmentation

**SENTINEL-CXR** — Uncertainty-Aware Chest Radiograph Triage
Deep Learning (MAIB AI 114) · Prof Anshul Gupta · S P Jain School of Global Management, Dubai

| Group member | Student ID |
|---|---|
| Krishna Mathur | AS25DXB018 |
| Atharva Soundankar | AS25DXB020 |
| Yash Petkar | AS25DXB021 |

---

**Syllabus mapping — Week 6: Generative Adversarial Networks**

ChestX-ray14 is severely imbalanced — `Hernia` appears in under 0.3% of studies.
A DCGAN is trained on the rare classes and its samples are added to the training
set, with the change in **minority-class AUPRC** measured against an unaugmented
control.

The honest framing matters. Synthetic radiographs cannot add clinical information
the generator was never shown; at best they act as a learned regulariser. The
experiment is designed so the result can come out negative, and the notebook says
so if it does.


In [ ]:
# ── Environment ───────────────────────────────────────────────────────
# Runs on Colab free tier (T4). Nothing here needs a paid runtime.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "torchxrayvision", "scikit-learn", "seaborn"],
        check=False,
    )

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 20260812
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
})
INSTRUMENT, STAT = "#2E9CB8", "#D64541"

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────
# NIH ChestX-ray14: 112,120 frontal radiographs, 30,805 patients, 14 labels.
# Kaggle: https://www.kaggle.com/datasets/nih-chest-xrays/data
#
# In Colab, the fastest route is the Kaggle API:
#   from google.colab import files; files.upload()      # kaggle.json
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#   !kaggle datasets download -d nih-chest-xrays/data -p /content/nih --unzip

DATA_DIR = os.environ.get("NIH_DIR", "/content/nih")
META = os.path.join(DATA_DIR, "Data_Entry_2017.csv")

PATHOLOGIES = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Effusion",
               "Emphysema","Fibrosis","Hernia","Infiltration","Mass","Nodule",
               "Pleural_Thickening","Pneumonia","Pneumothorax"]

def load_metadata(path=META):
    """Load the label CSV and expand `Finding Labels` into 14 binary columns."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    for p in PATHOLOGIES:
        df[p] = df["Finding Labels"].str.contains(p, regex=False).astype(int)
    df["Patient Age"] = pd.to_numeric(df["Patient Age"], errors="coerce")
    # Ages above ~100 in this dataset are data-entry errors, not centenarians.
    df = df[(df["Patient Age"] > 0) & (df["Patient Age"] < 100)]
    return df

def patient_disjoint_split(df, fracs=(0.70, 0.10, 0.20), seed=SEED):
    """Split by Patient ID — NEVER by image.

    A patient contributes 3-4 follow-up studies. Splitting by image places the
    same patient's scans on both sides of the boundary, so the model can
    memorise the patient rather than the pathology. Every metric then reports a
    number that will not survive contact with a new hospital. This is the most
    common methodological error in published work on ChestX-ray14.
    """
    patients = df["Patient ID"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    a, b = int(fracs[0] * n), int((fracs[0] + fracs[1]) * n)
    sets = (set(patients[:a]), set(patients[a:b]), set(patients[b:]))
    train, cal, test = (df[df["Patient ID"].isin(s)].copy() for s in sets)
    assert not (set(train["Patient ID"]) & set(test["Patient ID"])), "patient leak"
    return train, cal, test

## 1. DCGAN following Radford et al.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent=100, base=64, ch=1):
        super().__init__(); self.latent = latent
        def blk(i, o, k, s, p):
            return nn.Sequential(nn.ConvTranspose2d(i, o, k, s, p, bias=False),
                                 nn.BatchNorm2d(o), nn.ReLU(True))
        self.net = nn.Sequential(
            blk(latent, base*8, 4, 1, 0), blk(base*8, base*4, 4, 2, 1),
            blk(base*4, base*2, 4, 2, 1), blk(base*2, base, 4, 2, 1),
            nn.ConvTranspose2d(base, ch, 4, 2, 1), nn.Tanh())
    def forward(self, z): return self.net(z.view(z.size(0), self.latent, 1, 1))

class Discriminator(nn.Module):
    def __init__(self, base=64, ch=1):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 4, 2, 1, bias=False),
                                 nn.BatchNorm2d(o), nn.LeakyReLU(0.2, True))
        self.net = nn.Sequential(
            nn.Conv2d(ch, base, 4, 2, 1), nn.LeakyReLU(0.2, True),
            blk(base, base*2), blk(base*2, base*4), blk(base*4, base*8),
            nn.Conv2d(base*8, 1, 4, 1, 0))
    def forward(self, x): return self.net(x).view(-1)   # logits

G, D = Generator().to(DEVICE), Discriminator().to(DEVICE)
print("G:", sum(p.numel() for p in G.parameters()), "| D:", sum(p.numel() for p in D.parameters()))

## 2. Training loop with stabilisation

GAN training is famously unstable. Three specific measures, each with a reason.

In [ ]:
def train_gan(loader, epochs=60, lr=2e-4, latent=100, label_smooth=0.9):
    """DCGAN training with three stabilisers:

    1. `BCEWithLogitsLoss` rather than sigmoid + BCE — numerically stable.
    2. One-sided label smoothing (real = 0.9, not 1.0). Stops the discriminator
       becoming over-confident, which otherwise starves the generator of gradient.
    3. betas=(0.5, 0.999). The default 0.9 momentum makes GAN training oscillate;
       Radford et al. found 0.5 necessary.
    """
    crit = nn.BCEWithLogitsLoss()
    optG = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    history = []
    for ep in range(epochs):
        dl = gl = 0.0
        for real, _ in loader:
            real = real.to(DEVICE); b = real.size(0)
            # ── discriminator
            optD.zero_grad(set_to_none=True)
            out_real = D(real)
            loss_real = crit(out_real, torch.full((b,), label_smooth, device=DEVICE))
            fake = G(torch.randn(b, latent, device=DEVICE))
            loss_fake = crit(D(fake.detach()), torch.zeros(b, device=DEVICE))
            (loss_real + loss_fake).backward(); optD.step()
            # ── generator: maximise log D(G(z)) rather than minimise log(1-D(G(z)))
            optG.zero_grad(set_to_none=True)
            loss_g = crit(D(fake), torch.ones(b, device=DEVICE))
            loss_g.backward(); optG.step()
            dl += (loss_real + loss_fake).item(); gl += loss_g.item()
        history.append((dl/len(loader), gl/len(loader)))
        if ep % 10 == 0: print(f"epoch {ep:3d}  D {history[-1][0]:.3f}  G {history[-1][1]:.3f}")
    return history

print("train_gan ready.")

## 3. The experiment that decides whether this helped

In [ ]:
def augmentation_experiment(train_df, cal_df, test_df, rare=("Hernia","Pneumonia","Emphysema")):
    """Control vs augmented, everything else identical.

    Reports AUPRC, not AUROC: for a label present in <1% of studies, AUROC is
    dominated by the vast negative class and barely moves even when minority
    performance changes a lot.
    """
    print("Protocol")
    print("  A. control   : train on real data only")
    print("  B. augmented : real + N synthetic samples for each rare class")
    print("  Identical seed, epochs, architecture, and TEST split.")
    print()
    print("Report delta AUPRC per rare class with a bootstrap CI. If the interval")
    print("crosses zero, the correct conclusion is that augmentation did not help,")
    print("and we report that. A negative result honestly reported is worth more")
    print("than a positive one obtained by tuning until the number improves.")
    return None

print("Experiment protocol defined.")

---

### References for this notebook

- Radford, A., Metz, L. & Chintala, S. (2015). Unsupervised representation learning with DCGANs. arXiv:1511.06434.
- Brock, A., Donahue, J. & Simonyan, K. (2018). Large scale GAN training. arXiv:1809.11096.
- Wang, T. et al. (2018). High-resolution image synthesis with conditional GANs. *CVPR*.

---

*SENTINEL-CXR is a student research prototype. It is not a medical device and
must not be used for clinical decisions.*
